# 🤖 Phase 5 — Transformers & BERT
## Simple, Clean, Error-Free Colab Notebook

---

**What you will learn:**
- What BERT is and why it is powerful
- Use BERT in 3 lines with Hugging Face `pipeline`
- Sentiment Analysis, NER, Question Answering, Fill-Mask
- Fine-tune DistilBERT on your own reviews

> **How to run:** `Runtime → Run all`  
> **Enable GPU first:** `Runtime → Change runtime type → T4 GPU`


---
## 📦 Step 0 — Install Transformers

Run this once. After it finishes, **restart the runtime**, then run everything else.

> `Runtime → Restart session` — then run from Step 1 downward.


In [1]:
!pip install transformers==4.44.2 accelerate>=0.26.0 --quiet

print('✅ Done! Now restart the runtime:')
print('   Runtime → Restart session')
print('   Then run from Step 1 downward.')


✅ Done! Now restart the runtime:
   Runtime → Restart session
   Then run from Step 1 downward.


---
## 🧠 Step 1 — What is BERT?

Old models (like LSTM) read words **one by one** → left to right.

BERT reads **all words at the same time** and looks at every word's relationship with every other word.

```
Sentence: 'The bank by the river was flooded'

LSTM sees:  The → bank → by → the → river ...  (left to right only)
BERT sees:  bank ↔ river ↔ flooded ↔ the  (all directions at once!)
```

This is called **Attention** — BERT pays attention to the whole sentence at once.

**DistilBERT** = a smaller, faster version of BERT — 40% smaller, same quality.


In [2]:
# Just check imports are working — no output expected except the print
import torch
from transformers import pipeline
import transformers

print(f'Transformers : {transformers.__version__}')
print(f'PyTorch      : {torch.__version__}')
print(f'Device       : {"GPU ✅" if torch.cuda.is_available() else "CPU (works fine)"}')


Transformers : 4.44.2
PyTorch      : 2.10.0+cu128
Device       : GPU ✅


---
## 😊😞 Step 2 — Sentiment Analysis in 3 Lines

`pipeline()` downloads a pre-trained model and wraps it so you can use it immediately.

No training needed — BERT already learned from millions of texts!


In [3]:
# Load the sentiment analysis pipeline
# First run downloads ~250 MB — cached after that
sentiment = pipeline('sentiment-analysis')

# Test it!
reviews = [
    'This movie was absolutely amazing!',
    'I hated every minute of this film.',
    'It was okay, nothing special.',
    'One of the best movies I have ever seen!',
    'Terrible acting and a boring story.',
]

print('SENTIMENT ANALYSIS RESULTS')
print('=' * 45)
for r in reviews:
    result = sentiment(r)[0]
    emoji  = '😊' if result['label'] == 'POSITIVE' else '😞'
    print(f'{emoji} {result["label"]:8s}  ({result["score"]:.0%})  "{r}"')


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


SENTIMENT ANALYSIS RESULTS
😊 POSITIVE  (100%)  "This movie was absolutely amazing!"
😞 NEGATIVE  (100%)  "I hated every minute of this film."
😞 NEGATIVE  (98%)  "It was okay, nothing special."
😊 POSITIVE  (100%)  "One of the best movies I have ever seen!"
😞 NEGATIVE  (100%)  "Terrible acting and a boring story."


---
## 🏷️ Step 3 — Named Entity Recognition (NER)

NER finds **names of people, places, and organisations** in a sentence automatically.

```
'Elon Musk founded Tesla in California'
           ↑ PERSON  ↑ ORG     ↑ LOCATION
```


In [4]:
# Load the NER pipeline
ner = pipeline('ner', aggregation_strategy='simple')

sentence = 'Elon Musk founded Tesla in California and later started SpaceX in Hawthorne.'

results = ner(sentence)

print('Named Entities found:')
print('=' * 40)
for entity in results:
    print(f"  {entity['entity_group']:10s}  →  {entity['word']}")


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision f2482bf (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Named Entities found:
  PER         →  Elon Musk
  ORG         →  Tesla
  LOC         →  California
  ORG         →  SpaceX
  LOC         →  Hawthorne


---
## ❓ Step 4 — Question Answering

Give BERT a **context paragraph** and a **question** — it finds the answer inside the text.

This is how search engines and chatbots extract facts from documents!


In [5]:
# Load the question answering pipeline
qa = pipeline('question-answering')

context = """
DistilBERT is a smaller and faster version of BERT.
It was created by Hugging Face in 2019.
DistilBERT has 66 million parameters and runs 60 percent faster than BERT.
It keeps 97 percent of BERT's accuracy on most tasks.
"""

questions = [
    'Who created DistilBERT?',
    'How many parameters does DistilBERT have?',
    'When was DistilBERT created?',
]

print('QUESTION ANSWERING')
print('=' * 45)
for q in questions:
    answer = qa(question=q, context=context)
    print(f'  Q: {q}')
    print(f'  A: {answer["answer"]}  (confidence: {answer["score"]:.0%})')
    print()


No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


QUESTION ANSWERING
  Q: Who created DistilBERT?
  A: Hugging Face  (confidence: 100%)

  Q: How many parameters does DistilBERT have?
  A: 66 million  (confidence: 88%)

  Q: When was DistilBERT created?
  A: 2019  (confidence: 95%)



---
## 🎭 Step 5 — Fill-Mask

Hide a word with `[MASK]` and BERT guesses what word fits best.

This is actually **how BERT was trained** — it learned by filling in masked words from billions of sentences!


In [6]:
# Load the fill-mask pipeline — uses distilbert by default
fill = pipeline('fill-mask', model='distilbert-base-uncased')

sentences = [
    'The capital of France is [MASK].',
    'She studied [MASK] at university and became a doctor.',
    'The movie was so [MASK] that I watched it twice.',
]

print('FILL-MASK PREDICTIONS')
print('=' * 45)
for sentence in sentences:
    results = fill(sentence)
    print(f'Sentence : {sentence}')
    print(f'Top 3 guesses:')
    for r in results[:3]:
        print(f'   {r["score"]:.0%}  →  "{r["sequence"]}"')
    print()


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


FILL-MASK PREDICTIONS
Sentence : The capital of France is [MASK].
Top 3 guesses:
   14%  →  "the capital of france is marseille."
   9%  →  "the capital of france is nantes."
   9%  →  "the capital of france is toulouse."

Sentence : She studied [MASK] at university and became a doctor.
Top 3 guesses:
   54%  →  "she studied medicine at university and became a doctor."
   9%  →  "she studied law at university and became a doctor."
   3%  →  "she studied jurisprudence at university and became a doctor."

Sentence : The movie was so [MASK] that I watched it twice.
Top 3 guesses:
   17%  →  "the movie was so popular that i watched it twice."
   5%  →  "the movie was so great that i watched it twice."
   4%  →  "the movie was so successful that i watched it twice."



---
## 🎯 Step 6 — Fine-Tuning DistilBERT on Your Own Data

**Pre-training** = BERT learned from the whole internet (done by researchers, takes weeks).

**Fine-tuning** = We take that trained BERT and teach it our specific task in a few minutes.

```
Pre-trained BERT  →  add classifier layer  →  train on our reviews  →  our custom model
    (frozen knowledge)      (new head)           (a few minutes)          (ready!)
```

We only need **a few minutes and a small dataset** because BERT already knows language!


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE.upper()}')


Using: CUDA


### 6a — Our training data

16 movie reviews — 8 positive, 8 negative.


In [8]:
reviews = [
    # Positive (label = 1)
    'this movie was absolutely amazing and wonderful',
    'i loved every minute of this film it was great',
    'fantastic story with brilliant acting and direction',
    'one of the best movies i have ever seen in my life',
    'truly an outstanding film with superb performances',
    'the movie was so good i watched it twice already',
    'excellent cinematography and a gripping storyline',
    'a masterpiece of cinema that everyone should watch',
    # Negative (label = 0)
    'this movie was absolutely terrible and boring to watch',
    'i hated this film it was a complete waste of time',
    'worst movie i have ever seen in my entire life',
    'terrible acting and a very predictable boring story',
    'i fell asleep during this dull and disappointing film',
    'the movie was so bad i left the cinema early tonight',
    'awful direction and a pointless story with no ending',
    'do not waste your money on this horrible film at all',
]
labels = [1]*8 + [0]*8

print(f'Reviews: {len(reviews)}  |  Positive: {labels.count(1)}  |  Negative: {labels.count(0)}')


Reviews: 16  |  Positive: 8  |  Negative: 8


### 6b — Tokenize + Dataset class


In [9]:
MODEL = 'distilbert-base-uncased'

# Tokenizer converts words to numbers
tokenizer = AutoTokenizer.from_pretrained(MODEL)
encodings = tokenizer(reviews, truncation=True, padding=True, max_length=64, return_tensors='pt')

# PyTorch Dataset wrapper
class ReviewDataset(Dataset):
    def __init__(self, enc, lab):
        self.enc = enc
        self.lab = lab
    def __len__(self):
        return len(self.lab)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.lab[i])
        return item

# 12 for training, 4 for testing
train_data = ReviewDataset({k: v[:12] for k,v in encodings.items()}, labels[:12])
test_data  = ReviewDataset({k: v[12:] for k,v in encodings.items()}, labels[12:])

print(f'Train: {len(train_data)} samples  |  Test: {len(test_data)} samples')


Train: 12 samples  |  Test: 4 samples


### 6c — Load model and train


In [10]:
# Load DistilBERT with a 2-class classifier head
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model = model.to(DEVICE)

# Training settings
args = TrainingArguments(
    output_dir        = './results',
    num_train_epochs  = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    eval_strategy     = 'epoch',   # ✅ correct name (not evaluation_strategy)
    learning_rate     = 2e-5,
    save_strategy     = 'no',
    report_to         = 'none',
)

trainer = Trainer(
    model         = model,
    args          = args,
    train_dataset = train_data,
    eval_dataset  = test_data,
)

print('Training...')
trainer.train()
print('✅ Done!')


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training...


Epoch,Training Loss,Validation Loss
1,No log,0.764563
2,No log,0.796507
3,No log,0.809317


✅ Done!


### 6d — Predict on new reviews


In [11]:
def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    label = 'POSITIVE 😊' if probs[1] > probs[0] else 'NEGATIVE 😞'
    conf  = max(probs).item()
    print(f'{label}  ({conf:.0%})  — "{text}"')

print('NEW REVIEW PREDICTIONS')
print('=' * 55)
predict('This film was a truly incredible experience!')
predict('I hated every second of this boring terrible movie.')
predict('Stunning visuals and a brilliant cast throughout.')
predict('Absolutely dreadful. Worst film I have ever watched.')


NEW REVIEW PREDICTIONS
POSITIVE 😊  (58%)  — "This film was a truly incredible experience!"
POSITIVE 😊  (56%)  — "I hated every second of this boring terrible movie."
POSITIVE 😊  (59%)  — "Stunning visuals and a brilliant cast throughout."
POSITIVE 😊  (56%)  — "Absolutely dreadful. Worst film I have ever watched."


### ✏️ Try your own review!


In [12]:
# Change this to any movie review and run the cell!
my_review = 'I loved this movie so much it made me cry!'
predict(my_review)


POSITIVE 😊  (58%)  — "I loved this movie so much it made me cry!"


---
## ✅ Summary — What You Learned

| Step | Topic | Key idea |
|---|---|---|
| 1 | What is BERT | Reads all words at once using Attention |
| 2 | Sentiment pipeline | Use BERT in 3 lines — no training needed |
| 3 | NER pipeline | Finds people, places, organisations in text |
| 4 | Question Answering | BERT extracts answers from a paragraph |
| 5 | Fill-Mask | BERT guesses missing words (how it was trained) |
| 6 | Fine-tuning | Adapt pre-trained BERT to your own task in minutes |

---

### 🔑 3 things to remember
1. **Pre-trained** = BERT already knows language. You do not train from scratch.
2. **Fine-tuning** = You add a small layer on top and train briefly on your data.
3. **`pipeline()`** = The easiest way to use any BERT model in just a few lines.

---
*Phase 5 complete! 🎉*
